# Week 14 — Offline RL Workbook

This workbook has two explicit exercise modes, controlled by `REFERENCE_MODE`
in the setup cell.

- **Reference (`True`, default):** supplied implementations demonstrate the
  calculations. Passing these checks verifies the reference, not your code.
- **Learner (`False`):** complete each scaffold's function, then restart and
  run all cells. The corresponding checks call your function. An unfinished
  scaffold raises `NotImplementedError`; there is no reference fallback.
  In a reference-wrapper scaffold, replace its guarded placeholder body with
  your implementation before choosing learner mode.

Change the setup flag before restarting. Record a boundary-case prediction
before each check and change an input after it passes. Later experiment cells
without a scaffold are reference demonstrations, not automated learner grades.

**Teaching companion:** [Worked examples, assumptions and self-checks](teaching_guide.md).
For additional independently graded tasks, see the
[progressive coding projects](../docs/INTERACTIVE_LEARNING.md).


**A bounded first investigation:** Make unsupported optimism visible. Start with **Exercises 1, 2 and 3** and the
[guide’s prediction, visual diagnostic and changed-input check](teaching_guide.md#notebook-investigation-make-unsupported-optimism-visible).
Save your own prediction before running the reference cells. For your own implementation, choose learner mode in setup, restart, and verify
that the corresponding check calls your completed function.


**Teaching companion:** [Worked examples, derivations, analogies, and self-checks](teaching_guide.md). Read the relevant example before attempting its exercise.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath('.'))
from common.plotting import use_rl_style, RL_PALETTE
from common.seed import seed_everything
use_rl_style(); seed_everything(0)
import numpy as np, matplotlib.pyplot as plt
from project import solutions as S
# True runs documented references. False checks YOUR completed functions.
# Restart and run all after changing this flag; an unfinished learner stops.
REFERENCE_MODE = True
def select_exercise(reference, learner):
    return reference if REFERENCE_MODE else learner
print("Exercise mode:", "reference demonstration" if REFERENCE_MODE else "learner checks (no fallback)")


## Exercise 1 (★) — Expectile loss

> **Hint** (try yourself first): The expectile loss is `|tau - 1{u<0}| * u^2`. For `tau > 0.5` it penalises positive residuals (under-prediction) more heavily.

In [ ]:
def expectile_loss(u, tau):
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:
expectile_loss = select_exercise(S.exercise_1_expectile, expectile_loss)
u = np.array([-1.0, -0.5, 0.0, 0.5, 1.0])
out = expectile_loss(u, 0.8)
assert out.shape == u.shape and (out >= 0).all() and out[-1] > out[0]
np.testing.assert_allclose(out, np.array([.2,.05,0.,.2,.8]))


## Exercise 2 (★) — AWR weights

> **Hint**: $w_i \propto \exp(\beta A_i)$; cap raw weights before normalising; shift log weights to avoid overflow; normalise so the weights average to 1.

In [ ]:
def awr_weights(adv, beta):
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:
awr_weights = select_exercise(S.exercise_2_awr_weights, awr_weights)
w = awr_weights(np.array([-1.0, 0.0, 1.0]), beta=2.0)
assert abs(w.mean() - 1.0) < 1e-6
raw_expected=np.array([np.exp(-2),1.,np.exp(2)])
np.testing.assert_allclose(w, raw_expected/raw_expected.mean())


## Exercise 3 (★★) — CQL penalty

> **Hint**: For a fixed state $s$, the penalty is `logsumexp(Q[s,:]) - Q[s, a_beta]`. For two or more finite Q-values it is strictly positive. It tends to zero only as the logged action dominates the softmax (or equals zero with a single available action).

In [ ]:
def cql_penalty(Q_s, a_beta):
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:
cql_penalty = select_exercise(S.exercise_3_cql_penalty, cql_penalty)
Q = np.array([1.0, 0.0, 0.0, 0.0])
assert cql_penalty(Q, 0) < cql_penalty(Q, 3)
assert np.isclose(cql_penalty(Q,0), np.log(np.e+3)-1)
assert np.isclose(cql_penalty(Q,3), np.log(np.e+3))


## Exercise 4 (★★) — Collect the logged dataset

> **Hint**: Use the `collect_logged_dataset` helper. Inspect coverage by counting visits per (s,a). `episode_end` marks collection boundaries; `done` marks genuine termination for Bellman targets.

In [ ]:
def make_data(n_eps=80, seed=0):
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:
make_data = select_exercise(S.exercise_4_collect_dataset, make_data)
data = make_data(50)
assert data['s_idx'].shape == data['a'].shape

## Exercise 5 (★★) — Train IQL

> **Hint**: Use the reference `train` helper with `algo='iql'`. Plot the evaluation return curve.

In [ ]:
def train_iql(seed=0, n_steps=1200):
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:
train_iql = select_exercise(S.exercise_5_train_iql, train_iql)
out = train_iql(0, 500)
assert 'final_return' in out

## Exercise 6 (★★) — CQL ablation over $\alpha$

> **Hint**: Sweep $\alpha \in \{0.1, 1, 10\}$ and plot final return vs $\alpha$.

In [ ]:
def cql_ablation(alphas, n_steps=500):
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:
def reference_cql_ablation(alphas, n_steps=500):
    return {a: S.exercise_6_train_cql(0, n_steps, a)['final_return'] for a in alphas}
cql_ablation = select_exercise(reference_cql_ablation, cql_ablation)
res = cql_ablation([0.1, 1.0, 10.0], 300)
assert set(res.keys()) == {0.1, 1.0, 10.0}

## Exercise 7 (★★★) — SNIPS off-policy evaluation

> **Hint**: For each episode, accumulate per-step likelihood ratio `pi(a|s)/pi_beta(a|s)` and the discounted return. Split on `episode_end` and accumulate weights in log space. Then `sum(w*G)/sum(w)`; this evaluates discounted episode return for the explicitly smoothed target policy.

In [ ]:
def snips_estimate(data, target_actions, gamma=0.95):
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:
snips_estimate = select_exercise(S.exercise_7_off_policy_evaluation_is, snips_estimate)
data = select_exercise(S.exercise_4_collect_dataset, make_data)(50)
target = np.argmax(data['Q_star'], axis=1)
est = snips_estimate(data, target)
assert np.isfinite(est)

## Exercise 8 (★★★, meaningful objective) — Compare three algorithms

This exercise mirrors the recruitment-matching question: if you only had logged recruiter decisions and outcomes, which offline algorithm would you trust to rank candidates? Run all three on the same dataset and report final returns.

In [ ]:
def compare(seed=0, n_steps=800):
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:
compare = select_exercise(S.exercise_8_compare_algorithms, compare)
res = compare(0, 400)
print(res)
assert set(res.keys()) == {'naive', 'cql', 'iql'}

<!-- AUTO-HINTS-SECTION -->
## Hints toward answers

Attempt the exercise first. Before opening a hint, write the quantity you are
estimating, the data source, and the expected update direction. The
[teaching guide](teaching_guide.md) supplies worked examples, assumptions,
and transfer questions. The hints below collect the exercise's own inline
hints; they do not reveal or execute reference solutions.

<details>
<summary>Exercise 1 (★) — Expectile loss</summary>

> **Hint** (try yourself first): The expectile loss is `|tau - 1{u<0}| * u^2`. For `tau > 0.5` it penalises positive residuals (under-prediction) more heavily.

</details>

<details>
<summary>Exercise 2 (★) — AWR weights</summary>

> **Hint**: $w_i \propto \exp(\beta A_i)$; cap raw weights before normalising; shift log weights to avoid overflow; normalise so the weights average to 1.

</details>

<details>
<summary>Exercise 3 (★★) — CQL penalty</summary>

> **Hint**: For a fixed state $s$, the penalty is `logsumexp(Q[s,:]) - Q[s, a_beta]`. For two or more finite Q-values it is strictly positive. It tends to zero only as the logged action dominates the softmax (or equals zero with a single available action).

</details>

<details>
<summary>Exercise 4 (★★) — Collect the logged dataset</summary>

> **Hint**: Use the `collect_logged_dataset` helper. Inspect coverage by counting visits per (s,a). `episode_end` marks collection boundaries; `done` marks genuine termination for Bellman targets.

</details>

<details>
<summary>Exercise 5 (★★) — Train IQL</summary>

> **Hint**: Use the reference `train` helper with `algo='iql'`. Plot the evaluation return curve.

</details>

<details>
<summary>Exercise 6 (★★) — CQL ablation over $\alpha$</summary>

> **Hint**: Sweep $\alpha \in \{0.1, 1, 10\}$ and plot final return vs $\alpha$.

</details>

<details>
<summary>Exercise 7 (★★★) — SNIPS off-policy evaluation</summary>

> **Hint**: For each episode, accumulate per-step likelihood ratio `pi(a|s)/pi_beta(a|s)` and the discounted return. Split on `episode_end` and accumulate weights in log space. Then `sum(w*G)/sum(w)`; this evaluates discounted episode return for the explicitly smoothed target policy.

</details>

<details>
<summary>Exercise 8 (★★★, meaningful objective) — Compare three algorithms</summary>

No inline hint is supplied for this exercise. Use the corresponding derivation and code bridge in the teaching guide.

</details>

After solving, change one input or assumption and predict the result before
running it. In reference mode, passing checks verifies the demonstration. In learner mode,
the selected checks must pass against your completed functions without fallback.


## Statistical investigation — Offline support is an identification requirement

**Goal.** Construct two reward worlds that logged data cannot distinguish. This bounded reference investigation runs on CPU without downloads.
Write your prediction in a new Markdown cell before executing it.

**Predict.** The logger only chose actions 0 and 1. Two possible reward worlds agree on
those actions but disagree on action 2. Predict whether any amount of this
same logged data can determine the value of a target that always chooses 2.
A conservative objective can express preferences, but it cannot manufacture
the unobserved outcomes.

**Work a small case by hand.** The behavior distribution is (0.6,0.4,0), while the target is (0,0,1).
Both worlds give conditional reward means 0.4 and 0.6 on logged actions.
World A assigns action 2 mean 0 and world B assigns it mean 1. The observed
data distributions are identical, but target values are 0 and 1. Therefore
the target value is not a function of the observed data law alone.

**Reading bridge:** [Probability and statistics lesson](../docs/statistics/07-importance-weighting.md).
These are original course examples informed by the textbook sections in the
[statistics source catalog](../docs/statistics/SOURCES.md); textbook PDFs are not required to run them.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from common.plotting import RL_PALETTE, use_rl_style
use_rl_style()
rng_stats=np.random.default_rng(114)
logged_actions=rng_stats.choice([0,1],size=500,p=[.6,.4])
world_a=np.array([.4,.6,0.]);world_b=np.array([.4,.6,1.])
assert np.array_equal(world_a[logged_actions],world_b[logged_actions])
target=np.array([0.,0.,1.])
print('Target values compatible with identical logged mechanisms:',target@world_a,target@world_b)
print('Observed action counts:',np.bincount(logged_actions,minlength=3))
fig,axes=plt.subplots(1,2,figsize=(9,3.3))
axes[0].bar(np.arange(3),np.bincount(logged_actions,minlength=3));axes[0].set(xlabel='Logged action',ylabel='Observed rows')
axes[1].bar(np.arange(3)-.15,world_a,width=.3,label='World A')
axes[1].bar(np.arange(3)+.15,world_b,width=.3,label='World B')
axes[1].set(xlabel='Action',ylabel='Conditional mean reward',xticks=[0,1,2]);axes[1].legend()
fig.suptitle('More of the same log cannot distinguish these two worlds')
fig.tight_layout();plt.show()


**Interpret and transfer.** This is a constructive identification argument, not a noisy empirical
comparison. The missing action's outcome mechanism may take either value
without changing the observed data distribution. A model-based prediction
adds assumptions that must be stated.

**Transfer.** Give the logger probability 0.001 for action 2. Explain how the
problem changes from absent population support to rare-event uncertainty;
calculate the probability of observing zero action-2 rows in 500 draws.


**Read your result.** The empty action-2 bar is the relevant gap. The right-panel disagreement
concerns only that unobserved action. Narrower uncertainty about the other
bars cannot determine which world generated action-2 outcomes.

<details><summary>Worked feedback for the changed assumption</summary>

With logger probability 0.001 for action 2, 500 independent rows contain
0.5 action-2 observations in expectation. The chance of none is
0.999^500≈0.6064. Population support is now positive, so the worlds are
distinguishable with enough appropriate data; a finite sample with no such
row remains poorly informative. This differs from structural zero support.

</details>
